In [1]:
import os
import pandas as pd
import numpy as np

# 모달리티 비교: FP(fingerprint) / IMG(OCSR image) / SMI(ChemBERTa) / GNN(MGSSL) /
# Conf(ConforFormer 3D conformation) 각각을 단독으로 DownstreamMLP에 넣었을 때의 성능과,
# CUE(=CUE, softmax fusion으로 FP+IMG를 결합한 기존 파이프라인)를 비교한다.
# SMI/GNN/Conf는 cue4modal 저장소(cue4modal/smi_train.py, gnn_mgssl_train.py, conf_train.py)에서
# 임베딩을 캐싱한 뒤 DownstreamMLP만 새로 학습한 결과이며, FP/IMG/CUE(cmfe)와 동일한
# 5-fold 내부 CV·metrics.csv 스키마(train.py::aggregate_metrics)를 공유한다.
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
cue4modal_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue4modal/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 소스: (표시 이름, metrics.csv 루트 경로). CUE는 항상 마지막에 오도록 순서를 고정한다.
# FP/IMG/CUE는 cue_base, SMI/GNN/Conf는 cue4modal_base에서 가져온다.
model_sources = {
    'FP': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'FP'),
    'IMG': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'IMG'),
    'SMI': os.path.join(cue4modal_base, '{dataset}', '{target}', 'v1', 'SMI'),
    'GNN': os.path.join(cue4modal_base, '{dataset}', '{target}', 'v2', 'GNN_MGSSL'),
    'Conf': os.path.join(cue4modal_base, '{dataset}', '{target}', 'v1', 'CONF'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
}

model_order = ['FP', 'IMG', 'SMI', 'GNN', 'Conf', 'CUE']

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    metrics.csv가 아직 없으면(해당 모달리티를 아직 안 돌렸으면) None을 반환한다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
n_expected = len(datasets) * len(targets) * len(model_sources)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다 (기대값: {len(datasets)} x {len(targets)} x {len(model_sources)} = {n_expected}).")
if len(df_total) < n_expected:
    missing = n_expected - len(df_total)
    have = sorted(df_total['Model'].unique().tolist()) if not df_total.empty else []
    print(f"⚠️  아직 {missing}개 항목이 없습니다. 현재 확보된 모델: {have}")
    print("   (smi_train.py / gnn_train.py / conf_train.py 를 돌려 metrics.csv를 생성한 뒤 이 셀부터 다시 실행하면 됩니다.)")

✅ 데이터 로드 완료: 48개의 항목이 수집되었습니다 (기대값: 2 x 4 x 6 = 48).


In [3]:
METRIC_HEADERS = {'rmse': 'RMSE( ↓ )', 'r2': 'R2( ↑ )', 'pcc': 'PCC( ↑ )'}


def build_metric_table(df_all, dataset, target, metrics=metrics_of_interest, model_order=model_order):
    """지정한 dataset/target에 대해 Model별 metric 값('mean ± std')을 담은 표를 만든다 (순위 없음).

    Returns:
        Optional[pd.DataFrame]: 컴럼이 Model | RMSE( ↓ ) | R2( ↑ ) | PCC( ↑ ) 순서인 표.
            각 값은 '{mean:.3f} ± {std:.3f}' 문자열이다.
            해당 dataset/target에 데이터가 하나도 없으면 None. 일부 모달리티만 없으면
            해당 행만 제외된 표를 반환한다 (예: SMI/GNN/Conf 미실행 시 FP/IMG/CUE만 나옴).
    """
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target)]
    df_sub = df_sub.set_index('Model').reindex(model_order)
    df_sub = df_sub.dropna(subset=[m for m in metrics if m in df_sub.columns], how='all')
    if df_sub.empty:
        return None

    table = pd.DataFrame(index=df_sub.index)
    for m in metrics:
        if m not in df_sub.columns:
            continue
        header = METRIC_HEADERS.get(m, m.upper())
        if f'{m}_std' in df_sub.columns:
            table[header] = [f"{v:.3f} ± {s:.3f}" for v, s in zip(df_sub[m], df_sub[f'{m}_std'])]
        else:
            table[header] = df_sub[m].round(3)

    return table.reset_index()  # 'Model'을 첫 컴럼으로 복원


table_save_dir = os.path.join(cue_base, 'table', 'modal_comparison_v2')
os.makedirs(table_save_dir, exist_ok=True)

# DIS -> dis(dissociation), INH -> inh(inhibition)
dataset_filenames = {'DIS': 'dis.csv', 'INH': 'inh.csv'}

metric_tables = {}
for d_name in datasets:
    dataset_tables = []
    for target in targets:
        table = build_metric_table(df_total, d_name, target)
        if table is None:
            continue
        metric_tables[(d_name, target)] = table

        print(f"\n=== [{d_name} / {target}] ===")
        print(table.to_string(index=False))

        # 맨 처음 컴럼에 Target을 추가한 뒤(targets 순서 = PI, Ssel, WS2, RS2) 이어 붙인다.
        table_with_target = table.copy()
        table_with_target.insert(0, 'Target', target)
        dataset_tables.append(table_with_target)

    if not dataset_tables:
        continue

    dataset_table = pd.concat(dataset_tables, ignore_index=True)
    save_path = os.path.join(table_save_dir, dataset_filenames[d_name])
    dataset_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"\n✅ 저장 완료: {save_path}")


=== [Kd / PI] ===
Model     RMSE( ↓ )        R2( ↑ )      PCC( ↑ )
   FP 0.245 ± 0.012  0.098 ± 0.059 0.357 ± 0.085
  IMG 0.257 ± 0.007  0.008 ± 0.043 0.194 ± 0.084
  SMI 0.261 ± 0.006 -0.025 ± 0.071 0.118 ± 0.107
  GNN 0.269 ± 0.003 -0.090 ± 0.061 0.026 ± 0.062
 Conf 0.274 ± 0.018 -0.130 ± 0.117 0.122 ± 0.155
  CUE 0.242 ± 0.010  0.125 ± 0.055 0.366 ± 0.092

=== [Kd / Ssel] ===
Model     RMSE( ↓ )        R2( ↑ )      PCC( ↑ )
   FP 0.887 ± 0.042  0.081 ± 0.073 0.326 ± 0.072
  IMG 0.957 ± 0.105 -0.072 ± 0.188 0.189 ± 0.095
  SMI 0.909 ± 0.068  0.038 ± 0.070 0.211 ± 0.138
  GNN 0.968 ± 0.041 -0.094 ± 0.078 0.079 ± 0.099
 Conf 0.946 ± 0.033 -0.050 ± 0.119 0.201 ± 0.073
  CUE 0.895 ± 0.068  0.065 ± 0.107 0.314 ± 0.104

=== [Kd / WS(alpha=2)] ===
Model     RMSE( ↓ )        R2( ↑ )      PCC( ↑ )
   FP 0.285 ± 0.018  0.203 ± 0.052 0.470 ± 0.044
  IMG 0.300 ± 0.009  0.114 ± 0.098 0.380 ± 0.096
  SMI 0.321 ± 0.027 -0.014 ± 0.124 0.258 ± 0.135
  GNN 0.327 ± 0.026 -0.048 ± 0.126 0.134 ± 0.148
 

In [4]:
# metric별 rank 행렬: row = Dataset-Target(8개) + 마지막 'avg' 행,
# column = rank_model_order(FP/IMG/SMI/GNN/Conf). CUE는 fusion 결과라 단일 모달리티끼리의
# 순위 비교에서 제외한다.
rank_model_order = ['FP', 'IMG', 'SMI', 'GNN', 'Conf']


def build_rank_matrix(df_all, metric, model_order=rank_model_order):
    """지정한 metric에 대해 8개의 (Dataset, Target) 조합별로 model_order(CUE 제외, 5개
    단일 모달리티) 각 모델의 순위를 담은 행렬을 만든다. 마지막 'avg' 행은 각 모델의
    8개 행별 순위 평균이다.

    Returns:
        pd.DataFrame: 컬럼이 'Dataset-Target', FP, IMG, SMI, GNN, Conf 순인 표.
    """
    rows = {}
    for d_name in datasets:
        for target in targets:
            df_sub = df_all[(df_all['Dataset'] == d_name) & (df_all['Target'] == target)]
            df_sub = df_sub.set_index('Model').reindex(model_order)
            if metric not in df_sub.columns or df_sub[metric].isna().all():
                continue
            ranks = df_sub[metric].rank(ascending=LOWER_IS_BETTER.get(metric, True), method='min')
            rows[f'{d_name}-{target}'] = ranks

    rank_df = pd.DataFrame(rows).T
    rank_df = rank_df.reindex(columns=model_order)
    rank_df.loc['avg'] = rank_df.mean(axis=0).round(1)

    return rank_df.reset_index().rename(columns={'index': 'Dataset-Target'})


rank_table_save_dir = os.path.join(cue_base, 'table', 'modal_comparison_v2')
os.makedirs(rank_table_save_dir, exist_ok=True)

for metric in metrics_of_interest:
    rank_table = build_rank_matrix(df_total, metric)

    print(f"\n=== Rank table ({metric}) ===")
    print(rank_table.to_string(index=False))

    save_path = os.path.join(rank_table_save_dir, f'{metric}.csv')
    rank_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"✅ 저장 완료: {save_path}")


=== Rank table (rmse) ===
Dataset-Target  FP  IMG  SMI  GNN  Conf
         Kd-PI 1.0  2.0  3.0  4.0   5.0
       Kd-Ssel 1.0  4.0  2.0  5.0   3.0
Kd-WS(alpha=2) 1.0  2.0  4.0  5.0   3.0
    Kd-RS(k=2) 3.0  4.0  2.0  1.0   5.0
         Ki-PI 1.0  2.0  5.0  4.0   3.0
       Ki-Ssel 1.0  3.0  5.0  4.0   2.0
Ki-WS(alpha=2) 1.0  3.0  5.0  4.0   2.0
    Ki-RS(k=2) 1.0  2.0  4.0  5.0   3.0
           avg 1.2  2.8  3.8  4.0   3.2
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/table/modal_comparison_v2/rmse.csv

=== Rank table (r2) ===
Dataset-Target  FP  IMG  SMI  GNN  Conf
         Kd-PI 1.0  2.0  3.0  4.0   5.0
       Kd-Ssel 1.0  4.0  2.0  5.0   3.0
Kd-WS(alpha=2) 1.0  2.0  4.0  5.0   3.0
    Kd-RS(k=2) 3.0  4.0  2.0  1.0   5.0
         Ki-PI 1.0  2.0  5.0  4.0   3.0
       Ki-Ssel 1.0  3.0  5.0  4.0   2.0
Ki-WS(alpha=2) 1.0  3.0  5.0  4.0   2.0
    Ki-RS(k=2) 1.0  2.0  4.0  5.0   3.0
           avg 1.2  2.8  3.8  4.0   3.2
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/t